## Setup

In [ ]:
import pandas as pd

posts = pd.read_pickle("../pickles_tmp/3b-first-date_posts-all.pkl")
posts.shape

## Keyword distribution by subreddit
Stacked bar showing which subreddits contribute to each first-date keyword (normalized to 100%).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import ast

df_plot = posts.copy()
fontsize = 13


def ensure_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [x]
    return x if isinstance(x, list) else []


df_plot['extraction_keywords'] = df_plot['extraction_keywords'].apply(ensure_list)
df_exploded = df_plot.explode('extraction_keywords')
df_exploded = df_exploded[df_exploded['extraction_keywords'].str.strip() != ""]
df_exploded['subreddit'] = df_exploded['subreddit'].apply(
    lambda x: f"r/{x}" if not str(x).startswith('r/') else x
)
# Calculate counts and sort
counts = df_exploded.groupby(['extraction_keywords', 'subreddit']).size().unstack(fill_value=0)

# Sort: Largest total volume at the TOP of the horizontal chart
total_order = counts.sum(axis=1).sort_values(ascending=True).index
counts = counts.loc[total_order]

# Convert to 100% percentages
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 10))

# 'mako' or 'rocket' palettes are naturally more muted than standard bright colors
# alpha=0.8 adds a slight transparency to further "dim" the colors
colors = sns.color_palette("mako", n_colors=len(percentages.columns))

percentages.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=colors,
    edgecolor='white',
    width=0.75,
    alpha=0.8  # Dim the brightness
)

# Add percentage text inside bars
plt.yticks(fontsize=fontsize)
plt.xticks(fontsize=fontsize)
for p in ax.patches:
    w = p.get_width()
    if w > 7:  # Only label segments > 7% to keep it clean
        ax.text(
            p.get_x() + w / 2.,
            p.get_y() + p.get_height() / 2.,
            f'{int(w)}%',
            ha='center', va='center',
            color='white', fontweight='bold', fontsize=fontsize - 1
        )

# plt.title('Subreddit Contribution to Keywords (Percentage)', fontsize=16, fontweight='bold', loc='left', pad=20)
plt.xlabel('%', fontsize=fontsize)
plt.ylabel('', fontsize=fontsize, fontweight='bold')
plt.xlim(0, 100)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=fontsize)
plt.grid(axis='x', linestyle='--', alpha=0.3)  # Subtle grid

plt.tight_layout()
plt.show()

## Corpus composition by subreddit
Pie chart of total post share per subreddit.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

fontsize = 20

# 1. Prepare the Data
df_pie = posts.copy()

# Ensure "r/" prefix is present for all entries
df_pie['subreddit'] = df_pie['subreddit'].apply(
    lambda x: f"r/{x}" if not str(x).startswith('r/') else x
)

# Get the count of documents per subreddit
subreddit_counts = df_pie['subreddit'].value_counts()

# 2. Setup the Visuals
fig, ax = plt.subplots(figsize=(12, 12))

# Using the dimmed 'mako' palette with an alpha to keep it professional
colors = sns.color_palette("mako", n_colors=len(subreddit_counts))

# 3. Create the Pie Chart
wedges, texts, autotexts = ax.pie(
    subreddit_counts,
    labels=subreddit_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=1.2,
    labeldistance=1.3,
    textprops={'fontsize': fontsize},  # Subreddit name font size
    wedgeprops={'edgecolor': 'white', 'linewidth': 2, 'alpha': 0.75}  # Dimmed effect
)

# 4. Refine Label Styles
for autotext in autotexts:
    autotext.set_fontsize(fontsize)
    autotext.set_fontweight('bold')
    autotext.set_color('#333333')  # Dark grey for a cleaner look

for text in texts:
    text.set_fontsize(fontsize)

# 5. Final Formatting
# plt.title('Total Dataset Composition by Subreddit', fontsize=18, fontweight='bold', pad=50)

# Ensures the chart is a circle and labels have enough breathing room
ax.axis('equal')
plt.tight_layout()
plt.show()

## Posting activity over time
Quarterly stacked bar chart showing submission volume per subreddit.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ==========================================
# 1. PREPARE DATA
# ==========================================
df_bin = posts.copy()

# Convert UTC to datetime
df_bin['created_utc'] = pd.to_datetime(df_bin['created_utc'], unit='s')

# Ensure "r/" prefix for all subreddits
df_bin['subreddit'] = df_bin['subreddit'].apply(
    lambda x: f"r/{x}" if not str(x).startswith('r/') else x
)

# BINNING: Create a 'Quarter' column for macro-level analysis
# This groups months together to reduce the number of data points from 12 per year to 4
df_bin['Quarter'] = df_bin['created_utc'].dt.to_period('Q').astype(str)

# Group by Quarter and Subreddit
binned_data = df_bin.groupby(['Quarter', 'subreddit']).size().unstack(fill_value=0)

# ==========================================
# 2. PLOT BINNED BAR CHART
# ==========================================
fig, ax = plt.subplots(figsize=(14, 8))

# Use the dimmed 'mako' palette
colors = sns.color_palette("mako", n_colors=len(binned_data.columns))

# Plotting as a stacked bar chart for a cleaner look at distribution
binned_data.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=colors,
    edgecolor='white',
    width=0.8,
    alpha=0.75  # Muted brightness
)
tick_labels = []
tick_locations = []

for i, date in enumerate(binned_data.index):
    # If it's the first quarter of the year, mark it
    if i % 4 == 0:
        tick_labels.append(date)
        tick_locations.append(i)
    else:
        tick_labels.append("")
        tick_locations.append(i)

ax.set_xticks(tick_locations)
ax.set_xticklabels(tick_labels, rotation=45, fontsize=13)

# Set Y and X axis tick labels to Font Size 13
plt.yticks(fontsize=13)

# Legend adjustment: Font Size 13 and clear "r/" names
plt.legend(
    title_fontsize=13,
    fontsize=13,
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=False
)

ax.grid(False, axis='x')
plt.grid(axis='y', linestyle='--', alpha=0.2)
plt.xlabel('', fontsize=13)

plt.tight_layout()
plt.show()

## Author overview
Total submissions, distinct users, and average posts per user.

In [ ]:
# Calculate total unique authors
unique_authors_count = posts['author'].nunique()

# Calculate the ratio of posts per user
posts_per_user = len(posts) / unique_authors_count

print(f"Total Submissions: {len(posts)}")
print(f"Distinct Users: {unique_authors_count}")
print(f"Average Posts per User: {posts_per_user:.2f}")

## Power law of participation
Log-log scatter of submissions per author — confirms heavy-tailed "90-9-1" participation pattern.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ==========================================
# 1. PREPARE DATA
# ==========================================
# Count submissions per author
user_counts = posts['author'].value_counts()

# Calculate the frequency of those counts:
# x = Number of posts created (1, 2, 3...)
# y = Number of users who created exactly that many posts
freq_dist = user_counts.value_counts().sort_index()

x = freq_dist.index
y = freq_dist.values

# ==========================================
# 2. PLOT LOG-LOG SCATTERPLOT
# ==========================================
fig, ax = plt.subplots(figsize=(12, 8))

# Use a single dimmed color from the 'mako' palette
base_color = sns.color_palette("mako")[2]

ax.scatter(
    x, y,
    color=base_color,
    alpha=0.6,
    s=100,  # Larger points for clarity
    edgecolor='white',
    linewidth=1.5
)

# ==========================================
# 3. SET LOGARITHMIC SCALES (Critical for Social Computing)
# ==========================================
ax.set_xscale('log')
ax.set_yscale('log')

# ==========================================
# 4. THESIS FORMATTING (FONT 13)
# ==========================================
plt.title('Power Law of Participation: User Activity Distribution', fontsize=18, fontweight='bold', loc='left', pad=25)
plt.xlabel('Number of Submissions (Log Scale)', fontsize=14, fontweight='bold')
plt.ylabel('Number of Users (Log Scale)', fontsize=14, fontweight='bold')

# Set tick labels to Font Size 13
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)

# Add a subtle grid
plt.grid(True, which="both", ls="-", alpha=0.1)

# Annotations for your thesis
plt.text(x[0], y[0], f'  Most users ({y[0]:,})\n  post only once',
         fontsize=12, verticalalignment='bottom')

plt.tight_layout()
plt.show()

## Post length by subreddit
Boxplot (log scale) of word count per subreddit — measures narrative depth across communities.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import textstat  # Install via: pip install textstat
import numpy as np

# ==========================================
# 1. PRE-PROCESSING & CALCULATIONS
# ==========================================
df_text = posts.copy()

# Fill NaN values to avoid errors
# df_text['selftext'] = df_text['selftext'].fillna("")
# df_text['title'] = df_text['title'].fillna("")

# Calculate basic properties
df_text['word_count'] = df_text['selftext'].apply(lambda x: len(x.split()))
df_text['char_count'] = df_text['selftext'].apply(len)
df_text['sentence_count'] = df_text['selftext'].apply(textstat.sentence_count)

# Calculate Readability (Flesch Reading Ease)
# Higher score = easier to read. Standard for academic "accessibility" checks.
df_text['readability'] = df_text['selftext'].apply(
    lambda x: textstat.flesch_reading_ease(x) if len(x.split()) > 5 else np.nan
)

stats_summary = df_text[['word_count', 'sentence_count', 'readability']].describe().T
print("--- Textual Properties Summary ---")
print(stats_summary[['mean', 'std', 'min', '50%', 'max']])

plt.figure(figsize=(12, 7))

# Using a Boxplot to show "Effort" vs Subreddit
# We use log scale for word_count because Reddit posts vary from 1 to 5000+ words
sns.boxplot(
    data=df_text[df_text['word_count'] > 0],
    x='subreddit',
    y='word_count',
    palette="mako",
    showfliers=False  # Removes extreme outliers for a cleaner thesis graphic
)

plt.yscale('log')  # Log scale is vital for Social Computing distributions
# plt.title('Distribution of Post Length (Narrative Depth) by Subreddit', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('', fontsize=13)
plt.xlabel('', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=13)
plt.yticks(fontsize=13)
plt.tight_layout()
plt.show()

## Power-law distributions (3-panel)
Log-log scatter of submissions per author, score per post, and comments per post.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Set the style for thesis consistency
sns.set_theme(style="white")
mako_colors = sns.color_palette("mako", n_colors=3)

# Create a figure with 3 small subplots stacked vertically
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 14))

# ---------------------------------------------------------
# 1. POWER LAW OF ACTIVITY (Submissions per Author)
# ---------------------------------------------------------
user_counts = posts['author'].value_counts()
freq_dist = user_counts.value_counts().sort_index()
ax1.scatter(freq_dist.index, freq_dist.values, color=mako_colors[0], alpha=0.6, s=40, edgecolor='w')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_title('Submissions per Author', fontsize=15, fontweight='bold', loc='center')
ax1.set_xlabel('Posts Created', fontsize=13)
ax1.set_ylabel('Frequency', fontsize=13)

# ---------------------------------------------------------
# 2. POWER LAW OF VALIDATION (Score per Post)
# ---------------------------------------------------------
# Filter > 0 for log-log plot
score_dist = posts[posts['score'] > 0]['score'].value_counts().sort_index()
ax2.scatter(score_dist.index, score_dist.values, color=mako_colors[1], alpha=0.6, s=40, edgecolor='w')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_title('Score per Post', fontsize=15, fontweight='bold', loc='center')
ax2.set_xlabel('Upvotes', fontsize=13)
ax2.set_ylabel('Frequency', fontsize=13)

# ---------------------------------------------------------
# 3. POWER LAW OF INTERACTION (Comments per Post)
# ---------------------------------------------------------
# Filter > 0 for log-log plot
comm_dist = posts[posts['num_comments'] > 0]['num_comments'].value_counts().sort_index()
ax3.scatter(comm_dist.index, comm_dist.values, color=mako_colors[2], alpha=0.6, s=40, edgecolor='w')
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.set_title('Comments per Post', fontsize=15, fontweight='bold', loc='center')
ax3.set_xlabel('Comments', fontsize=13)
ax3.set_ylabel('Frequency', fontsize=13)

# Thesis Formatting
for ax in [ax1, ax2, ax3]:
    ax.tick_params(labelsize=13)
    ax.grid(True, which="both", ls="-", alpha=0.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(pad=3.0)
plt.show()

# Statistics

## Word count statistics
Mean, median, std, skewness of post length; flags posts significantly longer than average.

In [ ]:
import pandas as pd
import numpy as np

# 1. Prepare Data
df_text = posts.copy()
df_text['selftext'] = df_text['selftext'].fillna("")

# 2. Calculate Word Count
df_text['word_count'] = df_text['selftext'].apply(lambda x: len(x.split()))

# 3. Generate Extended Statistics
# We use .agg to specifically get Mean, Median, and Std Dev
text_stats = df_text['word_count'].agg(['mean', 'median', 'std', 'min', 'max', 'count']).to_frame().T

# 4. Calculate the "Skewness" - A useful metric for Social Computing
# This tells you mathematically how 'leaning' your distribution is
text_stats['skewness'] = df_text['word_count'].skew()

print("--- Extended Textual Properties Summary ---")
print(text_stats.round(2).to_string(index=False))

# 5. Quick Check for the 'Power Law' in text
one_std_above = text_stats['mean'].values[0] + text_stats['std'].values[0]
long_narratives = len(df_text[df_text['word_count'] > one_std_above])
print(f"\nPosts significantly longer than average (> {one_std_above:.0f} words): {long_narratives} posts")

## Temporal scope
Earliest and latest submission dates in the corpus.

In [ ]:
import pandas as pd

# Convert to datetime and find the earliest entry
first_date_unix = posts['created_utc'].min()
first_date_readable = pd.to_datetime(first_date_unix, unit='s')

# Find the latest entry while we are at it for your 'Temporal Scope'
last_date_unix = posts['created_utc'].max()
last_date_readable = pd.to_datetime(last_date_unix, unit='s')

print(f"First submission date: {first_date_readable}")
print(f"Last submission date:  {last_date_readable}")
print(f"Total timespan:       {last_date_readable - first_date_readable}")

## Posting frequency per author
Breakdown of users by how many submissions they made (1, 2, 3+).

In [ ]:
# 1. Count how many submissions each unique author made
author_counts = posts['author'].value_counts()

# 2. Count the frequency of those submission totals
submission_frequency = author_counts.value_counts().sort_index()

# 3. Extract the specific counts for 1, 2, and 3 submissions
one_post_users = submission_frequency.get(1, 0)
two_post_users = submission_frequency.get(2, 0)
three_post_users = submission_frequency.get(3, 0)
total_users = len(author_counts)
three_plus_users = total_users - one_post_users - two_post_users

# 4. Display the results with percentages
print(f"Total Unique Users: {total_users:,}")
print("-" * 30)
print(f"Users with 1 submission: {one_post_users:,} ({one_post_users / total_users:.1%})")
print(f"Users with 2 submissions: {two_post_users:,} ({two_post_users / total_users:.1%})")
print(f"Users with 3+ submissions: {three_plus_users:,} ({three_plus_users / total_users:.1%})")